## Mapping available GPS data

This notebook produces a map of where the available GPS data exists.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import gzip
import pandas as pd
import numpy as np
import holoviews as hv
import geoviews as gv
import cartopy.crs as ccrs
import pyproj
import glob
from pathlib import Path
from tqdm import tqdm
from shapely import LineString
import pickle

from utig_radar_loading import file_util, stream_util, geo_util

In [ ]:
pd.options.mode.copy_on_write = True

hv.extension('bokeh')

### Single file

Test playground for messing with individual files. Feel free to skip this section.

In [ ]:
# ct_path = "/kucresis/scratch/data/UTIG/UTIG1/targ/pcor/ASB/JKB2d/R17Wb/AQNnr1/xds.gz"
# file = gzip.open(ct_path, 'rt')
# print(file.readline())
# ct_columns = ['prj', 'set', 'trn', 'seq', 'clk_y', 'clk_n', 'clk_d', 'clk_h', 'clk_m', 'clk_s', 'clk_f', 'tim']
# pd.read_csv(file, sep=r'\s+', names=ct_columns, index_col=False)

In [ ]:
# #gps_path = "/kucresis/scratch/data/UTIG/UTIG2/targ/pcor/AMY/JKB2u/X62c/GPSnc1/xds.gz"
# gps_path = "/kucresis/scratch/data/UTIG/UTIG1/targ/pcor/ALG/JKB0a/X01a/GPStp2/xds.gz"

# file = gzip.open(gps_path, 'rt')
# pd.read_csv(file, sep=r',', index_col=False, on_bad_lines='warn')

In [ ]:
# df = stream_util.load_gzipped_stream_file(gps_path, debug=True, parse=True, parse_kwargs={'use_ct': True})
# df

In [ ]:
# df_sub = df[['prj', 'set', 'trn', 'clk_y', 'LAT', 'LON']][::100]

# path = gv.Path(df_sub,
#                 ['LON', 'LAT'],
#                 ['prj', 'set', 'trn', 'clk_y'],
#                 crs=ccrs.PlateCarree()
#               ).opts(
#                 projection=ccrs.SouthPolarStereo(),
#                 tools=['hover'],
#                 color='red',
#                 line_width=2
#               )

# stream_util.create_antarctica_basemap() * path

### BedMap Location Information
Load positioning information from BedMap as a reference for where we would expect data to be

In [ ]:
bedmap_csv_files = glob.glob("bedmap_utig/**/*UTIG*.csv")

bedmap_paths_cache_file = "outputs/bedmap_paths_cache.pkl"

if os.path.exists(bedmap_paths_cache_file):
    with open(bedmap_paths_cache_file, "rb") as f:
        bedmap_path_dfs = pickle.load(f)
else:
    bedmap_path_dfs = []

    for bedmap_path in tqdm(bedmap_csv_files):
        # Load the bedmap data
        df = pd.read_csv(bedmap_path, comment='#')[['longitude (degree_east)', 'latitude (degree_north)']]
        df = df.rename(columns={
            'longitude (degree_east)': 'LON',
            'latitude (degree_north)': 'LAT'
        })

        x_simplified, y_simplified = geo_util.project_split_and_simplify(df['LON'].values, df['LAT'].values)

        # Extract filename for tooltip
        filename = Path(bedmap_path).stem
        
        # Create dataframe with coordinates and tooltip info
        path_df = pd.DataFrame({
            'x': x_simplified,
            'y': y_simplified,
            'bedmap_src': filename
        })

        bedmap_path_dfs.append(path_df)

    with open(bedmap_paths_cache_file, "wb") as f:
        pickle.dump(bedmap_path_dfs, f)
    print(f"Saved cache of simplified BedMap paths to {bedmap_paths_cache_file}")


In [ ]:
bedmap_df_concat = pd.concat(bedmap_path_dfs, ignore_index=True)

bedmap_path = hv.Path(
                bedmap_df_concat,
                ['x', 'y'],
                ['bedmap_src'],
                label='BedMap (UTIG)'
                    ).opts(
                        tools=['hover'],
                        line_width=0.5,
                        line_dash='solid',
                        color='gray',
                        show_legend=True
                    )

In [ ]:
p = stream_util.create_antarctica_basemap() * bedmap_path
p = p.opts(aspect='equal', frame_width=500, frame_height=500, tools=['hover'])
p.opts(title='UTIG flight paths reported in BedMap')

### Multi-file

In [ ]:
use_cache = True
cache_dir = "outputs/file_index.csv"
base_path = "/kucresis/scratch/data/UTIG"

df_files = file_util.load_file_index_df(base_path, cache_dir, read_cache=use_cache)
df_artifacts = file_util.create_artifacts_df(df_files)

In [ ]:
df_files.head()

In [ ]:
radar_stream_types = [s for s in df_artifacts['stream'].unique() if "RAD" in s]

df_artifacts_gps = df_artifacts[df_artifacts['stream'].isin(['GPSnc1', 'GPStp2', 'GPSap1', 'GPSap3'])]
df_artifacts_gps = df_artifacts_gps[df_artifacts_gps['file_name'] == 'xds.gz']

df_artifacts_rad = df_artifacts[df_artifacts['stream'].isin(radar_stream_types)]

df_artifacts_gps['radar_path'] = None
df_artifacts_gps['radar_stream_type'] = None

for row_idx in df_artifacts_gps.index:
    # Find rows in df_artifacts_rad matching prj, set, and trn
    matching_rows = df_artifacts_rad[
        (df_artifacts_rad['prj'] == df_artifacts_gps.loc[row_idx, 'prj']) &
        (df_artifacts_rad['set'] == df_artifacts_gps.loc[row_idx, 'set']) &
        (df_artifacts_rad['trn'] == df_artifacts_gps.loc[row_idx, 'trn'])
    ]

    if (len(matching_rows) > 0) and (len(matching_rows['stream'].unique()) == 1):
        rad_artifact = df_artifacts_rad.iloc[0]
        df_artifacts_gps.loc[row_idx, 'radar_path'] = rad_artifact['full_path']
        df_artifacts_gps.loc[row_idx, 'radar_stream_type'] = rad_artifact['stream']
    elif len(matching_rows) > 1:
        print(f"Warning: Multiple matching radar rows found for GPS row {row_idx} with different stream types")
        print(matching_rows)

In [ ]:
#all_files_utig1 = list(df_artifacts[(df_artifacts['dataset'] == 'UTIG1') & (df_artifacts['stream'].isin(['GPSnc1', 'GPStp2', 'GPSap1']))]['full_path'])
#all_files_utig2 = list(df_artifacts[(df_artifacts['dataset'] == 'UTIG2') & (df_artifacts['stream'].isin(['GPSnc1', 'GPStp2', 'GPSap1']))]['full_path'])

df_artifacts_gps_utig1 = df_artifacts_gps[df_artifacts_gps['dataset'] == 'UTIG1']
df_artifacts_gps_utig2 = df_artifacts_gps[df_artifacts_gps['dataset'] == 'UTIG2']

In [ ]:
def load_gps_data(artifacts_df):
    segment_dfs = []
    file_paths_success = []
    file_paths_success_line_km = []
    file_paths_success_line_km_shapely = []
    file_paths_fail = []

    for _, row in tqdm(artifacts_df.iterrows(), total=len(artifacts_df)):

        f = row['full_path']
        if 'GPSkc1' in f:
            file_paths_fail.append(f)
            continue # GPSkc1 does not contain position information
        if 'GPSnc2' in f:
            file_paths_fail.append(f)
            continue # GPSnc2 does not contain position information
        if 'GPSgp1' in f:
            file_paths_fail.append(f)
            # It appears that all segments with a GPSgp1 file already have either a GPSnc1 or GPStp2 file that we can parse
            continue # No stream format information available for GPSgp1

        try:
            df = stream_util.load_xds_stream_file(f, debug=False, parse=True, parse_kwargs={'use_ct': True})

            line_length_km = stream_util.calculate_track_distance_km(df)

            _, _, line_length_m_shapely = geo_util.project_split_and_simplify(df['LON'].values, df['LAT'].values, calc_length=True)

            necessary_keys = ['prj', 'set', 'trn', 'clk_y', 'LAT', 'LON', 'TIMESTAMP']
            for k in necessary_keys:
                if k not in df:
                    df[k] = np.nan

            df_sub = df[['prj', 'set', 'trn', 'clk_y', 'LAT', 'LON', 'TIMESTAMP']]

            # Add radar stream type
            df_sub['radar_stream_type'] = row['radar_stream_type']

            segment_dfs.append(df_sub)

            file_paths_success.append(f)
            file_paths_success_line_km.append(line_length_km)
            file_paths_success_line_km_shapely.append(line_length_m_shapely / 1000)
        except Exception as e:
            file_paths_fail.append(f)
            print(f"Error loading {f}: {e}")

    print(f"Successfully loaded {len(file_paths_success)} files.")
    print(f"Failed to load {len(file_paths_fail)} files.")
    print(f"Skipped loading {len(artifacts_df) - len(file_paths_success) - len(file_paths_fail)} files.")

    return {
        'segment_dfs': segment_dfs,
        'file_paths_success': file_paths_success,
        'file_paths_success_line_km': file_paths_success_line_km,
        'file_paths_success_line_km_shapely': file_paths_success_line_km_shapely,
        'file_paths_fail': file_paths_fail
    }

print("== UTIG1 ==")
data_utig1 = load_gps_data(df_artifacts_gps_utig1)
print("== UTIG2 ==")
data_utig2 = load_gps_data(df_artifacts_gps_utig2)

In [ ]:
print(f"data_utig1: {np.array(data_utig1['file_paths_success_line_km_shapely']).sum()}")
print(f"data_utig2: {np.array(data_utig2['file_paths_success_line_km_shapely']).sum()}")
print(f"sum: {np.array(data_utig1['file_paths_success_line_km_shapely']).sum() + np.array(data_utig2['file_paths_success_line_km_shapely']).sum()}")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.hist(data_utig1["file_paths_success_line_km_shapely"], bins=30)
ax.grid()
# Add a vertical line at 50 km
ax.axvline(50, color='r', linestyle='--')
ax.set_xlabel('UTIG1 Transect Line Length [km]')
ax.set_ylabel('Counts')

In [ ]:
# import matplotlib.pyplot as plt
# fig, axs = plt.subplots(1, 3, figsize=(10, 4))
# axs[0].hist(data_utig2["file_paths_success_line_km"], bins=20)
# axs[1].hist(data_utig2["file_paths_success_line_km_shapely"], bins=20)
# axs[2].hist(np.array(data_utig2["file_paths_success_line_km"]) - np.array(data_utig2["file_paths_success_line_km_shapely"]), bins=20)

In [ ]:
print("== UTIG1 ==")
if len(data_utig1['segment_dfs']) > 0:
    dfs_utig1, path_utig1 = geo_util.create_path(data_utig1['segment_dfs'], path_opts_kwargs={'color': '#1167a2'})
else:
    dfs_utig1, path_utig1 = [], None
print(f"Generated {len(dfs_utig1)} flights from {len(data_utig1['segment_dfs'])} segments.")

print("== UTIG2 ==")
if len(data_utig2['segment_dfs']) > 0:
    dfs_utig2, path_utig2 = geo_util.create_path(data_utig2['segment_dfs'], path_opts_kwargs={'color': '#1167a2'})
else:
    dfs_utig2, path_utig2 = [], None
print(f"Generated {len(dfs_utig2)} flights from {len(data_utig2['segment_dfs'])} segments.")

In [ ]:
import geoviews.feature as gf
epsg_3031 = ccrs.Stereographic(central_latitude=-90, true_scale_latitude=-71)
coastline = gf.coastline.options(scale='50m').opts(projection=epsg_3031)

In [ ]:
# Plot
#p = stream_util.create_antarctica_basemap() #* bedmap_path
p = coastline
if path_utig1:
    p *= path_utig1.relabel('UTIG1')
if path_utig2:
    p *= path_utig2.relabel('UTIG2')
p = p.opts(legend_position='right', show_legend=False, click_policy='hide')
p = p.opts(aspect='equal')
p.opts(frame_width=500, frame_height=500)

In [ ]:
hv.save(p.opts(
                frame_height=600, frame_width=600
            ),
            'outputs/utig_gps_tmp.html')

In [ ]:
hv.save(p.opts(
                title='UTIG1 (orange) and UTIG2 (purple) GPS Tracks',
                frame_height=800, frame_width=800
            ),
            'outputs/utig_gps.html')

In [ ]:
line_km = np.array(data_utig1['file_paths_success_line_km'])
line_km = line_km[~np.isnan(line_km)]
print(f"UTIG1: {line_km.sum().item():.1f} line km")

line_km = np.array(data_utig2['file_paths_success_line_km'])
line_km = line_km[~np.isnan(line_km)]
print(f"UTIG2: {line_km.sum().item():.1f} line km")

### Scratch

In [ ]:
# file_paths_fail = data_utig2['file_paths_fail']
# file_paths_success = data_utig2['file_paths_success']
# file_paths_success_line_km = data_utig2['file_paths_success_line_km']

# # Files that failed to load
# failed_files = pd.DataFrame(
#       [Path(fp).parts[-5:-1] for fp in file_paths_fail],
#       columns=['campaign', 'flight', 'segment', 'stream_type']
#   )
# failed_files['path'] = file_paths_fail
# failed_files['success'] = 0
# failed_files['line_km'] = 0

# # Successfully loaded files
# successful_files = pd.DataFrame(
#       [Path(fp).parts[-5:-1] for fp in file_paths_success],
#       columns=['campaign', 'flight', 'segment', 'stream_type']
#   )
# successful_files['path'] = file_paths_success
# successful_files['success'] = 1
# successful_files['line_km'] = file_paths_success_line_km

# merged_files = pd.concat([successful_files, failed_files], ignore_index=True)

# # Group by ['campaign', 'flight', 'segment'] and then sum success
# grouped = merged_files.groupby(['campaign', 'flight', 'segment']).sum().reset_index()

# print(f"Found a total of {grouped['line_km'].sum()} km")

# print(f"The following flights may not have loaded successfully:")
# grouped[grouped['success'] != 1]

In [ ]:
#grouped

In [ ]:
#successful_files